In [ ]:
# Locate the project when Jupyter starts in the root or notebooks directory.
from pathlib import Path
import os
import sys

project_root = next(
    folder for folder in (Path.cwd(), *Path.cwd().parents)
    if (folder / "project_paths.py").is_file()
)
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
from project_paths import DATA_DIR, MODELS_DIR, OUTPUT_DIR


In [2]:
from gpt_download import download_and_load_gpt2

settings, params = download_and_load_gpt2(
    model_size="124M",models_dir=MODELS_DIR
)

File already exists and is up-to-date: gpt2\124M\checkpoint
File already exists and is up-to-date: gpt2\124M\encoder.json
File already exists and is up-to-date: gpt2\124M\hparams.json
File already exists and is up-to-date: gpt2\124M\model.ckpt.data-00000-of-00001
File already exists and is up-to-date: gpt2\124M\model.ckpt.index
File already exists and is up-to-date: gpt2\124M\model.ckpt.meta
File already exists and is up-to-date: gpt2\124M\vocab.bpe


In [3]:
print("Settings:", settings)
print("Parameter dictionary keys:", params.keys())

Settings: {'n_vocab': 50257, 'n_ctx': 1024, 'n_embd': 768, 'n_head': 12, 'n_layer': 12}
Parameter dictionary keys: dict_keys(['blocks', 'b', 'g', 'wpe', 'wte'])


In [4]:
print(params["wte"])
print("Token embedding weight tensor dimensions:", params["wte"].shape)

[[-0.11010301 -0.03926672  0.03310751 ... -0.1363697   0.01506208
   0.04531523]
 [ 0.04034033 -0.04861503  0.04624869 ...  0.08605453  0.00253983
   0.04318958]
 [-0.12746179  0.04793796  0.18410145 ...  0.08991534 -0.12972379
  -0.08785918]
 ...
 [-0.04453601 -0.05483596  0.01225674 ...  0.10435229  0.09783269
  -0.06952604]
 [ 0.1860082   0.01665728  0.04611587 ... -0.09625227  0.07847701
  -0.02245961]
 [ 0.05135201 -0.02768905  0.0499369  ...  0.00704835  0.15519823
   0.12067825]]
Token embedding weight tensor dimensions: (50257, 768)


In [5]:
model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

In [6]:
model_name = "gpt2-small (124M)"
GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 1024,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias":False,
}
NEW_CONFIG = GPT_CONFIG_124M.copy()
NEW_CONFIG.update(model_configs[model_name])

In [7]:
NEW_CONFIG.update({"context_length": 1024})

In [10]:

NEW_CONFIG.update({"qkv_bias": True})
from chapter04 import GPTModel
import torch

gpt = GPTModel(NEW_CONFIG)
gpt.eval()

def assign(left, right):
    if left.shape != right.shape:
        raise ValueError(
            f"Shape mismatch. Left: {left.shape}, "
            f"Right: {right.shape}"
        )
    return torch.nn.Parameter(torch.tensor(right))

In [11]:
import numpy as np

def load_weights_into_gpt(gpt, params):
    gpt.pos_emb.weight = assign(gpt.pos_emb.weight, params["wpe"])
    gpt.tok_emb.weight = assign(gpt.tok_emb.weight, params["wte"])

    for b in range(len(params["blocks"])):
        q_w, k_w, v_w = np.split(
            params["blocks"][b]["attn"]["c_attn"]["w"], 3, axis=-1
        )
        gpt.trf_blocks[b].att.W_query.weight = assign(
            gpt.trf_blocks[b].att.W_query.weight, q_w.T)
        gpt.trf_blocks[b].att.W_key.weight = assign(
            gpt.trf_blocks[b].att.W_key.weight, k_w.T)
        gpt.trf_blocks[b].att.W_value.weight = assign(
            gpt.trf_blocks[b].att.W_value.weight, v_w.T)

        q_b, k_b, v_b = np.split(
            params["blocks"][b]["attn"]["c_attn"]["b"], 3, axis=-1
        )
        gpt.trf_blocks[b].att.W_query.bias = assign(
            gpt.trf_blocks[b].att.W_query.bias, q_b)
        gpt.trf_blocks[b].att.W_key.bias = assign(
            gpt.trf_blocks[b].att.W_key.bias, k_b)
        gpt.trf_blocks[b].att.W_value.bias = assign(
            gpt.trf_blocks[b].att.W_value.bias, v_b)

        gpt.trf_blocks[b].att.out_proj.weight = assign(
            gpt.trf_blocks[b].att.out_proj.weight,
            params["blocks"][b]["attn"]["c_proj"]["w"].T)
        gpt.trf_blocks[b].att.out_proj.bias = assign(
            gpt.trf_blocks[b].att.out_proj.bias,
            params["blocks"][b]["attn"]["c_proj"]["b"])

        gpt.trf_blocks[b].ff.layers[0].weight = assign(
            gpt.trf_blocks[b].ff.layers[0].weight,
            params["blocks"][b]["mlp"]["c_fc"]["w"].T)
        gpt.trf_blocks[b].ff.layers[0].bias = assign(
            gpt.trf_blocks[b].ff.layers[0].bias,
            params["blocks"][b]["mlp"]["c_fc"]["b"])
        gpt.trf_blocks[b].ff.layers[2].weight = assign(
            gpt.trf_blocks[b].ff.layers[2].weight,
            params["blocks"][b]["mlp"]["c_proj"]["w"].T)
        gpt.trf_blocks[b].ff.layers[2].bias = assign(
            gpt.trf_blocks[b].ff.layers[2].bias,
            params["blocks"][b]["mlp"]["c_proj"]["b"])

        gpt.trf_blocks[b].norm1.scale = assign(
            gpt.trf_blocks[b].norm1.scale,
            params["blocks"][b]["ln_1"]["g"])
        gpt.trf_blocks[b].norm1.shift = assign(
            gpt.trf_blocks[b].norm1.shift,
            params["blocks"][b]["ln_1"]["b"])
        gpt.trf_blocks[b].norm2.scale = assign(
            gpt.trf_blocks[b].norm2.scale,
            params["blocks"][b]["ln_2"]["g"])
        gpt.trf_blocks[b].norm2.shift = assign(
            gpt.trf_blocks[b].norm2.shift,
            params["blocks"][b]["ln_2"]["b"])

    gpt.final_norm.scale = assign(gpt.final_norm.scale, params["g"])
    gpt.final_norm.shift = assign(gpt.final_norm.shift, params["b"])
    gpt.out_head.weight = assign(gpt.out_head.weight, params["wte"])

In [12]:
load_weights_into_gpt(gpt,params)
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
gpt.to(device)


GPTModel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(1024, 768)
  (drop_emb): Dropout(p=0.1, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=True)
        (W_key): Linear(in_features=768, out_features=768, bias=True)
        (W_value): Linear(in_features=768, out_features=768, bias=True)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_shortcut): Dropout(p=0.1, inplace=False)
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=7

In [15]:
import torch
from chapter02 import create_dataloader_v1


# 1. 读取 The Verdict，按字符位置做 90% / 10% 划分
with open(DATA_DIR / "the-verdict.txt", "r", encoding="utf-8") as file:
    text_data = file.read()

split_idx = int(0.90 * len(text_data))
train_data = text_data[:split_idx]
val_data = text_data[split_idx:]

# 保留你原来的设置，每个样本 256 个词元，滑窗步长也是 256
torch.manual_seed(123)

train_loader = create_dataloader_v1(
    train_data,
    batch_size=2,
    max_length=256,
    stride=256,
    drop_last=True,
    shuffle=True,
    num_workers=0,
)

val_loader = create_dataloader_v1(
    val_data,
    batch_size=2,
    max_length=256,
    stride=256,
    drop_last=False,
    shuffle=False,
    num_workers=0,
)


# 2. 计算一个批次的交叉熵损失
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch = input_batch.to(device)
    target_batch = target_batch.to(device)

    logits = model(input_batch)

    # [B, T, 50257] -> [B*T, 50257]
    # [B, T]        -> [B*T]
    return torch.nn.functional.cross_entropy(
        logits.flatten(0, 1),
        target_batch.flatten(),
    )


# 3. 计算整个 DataLoader 的批次平均损失
def calc_loss_loader(data_loader, model, device, num_batches=None):
    if num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))

    if num_batches <= 0:
        return float("nan")

    total_loss = 0.0
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i >= num_batches:
            break
        loss = calc_loss_batch(input_batch, target_batch, model, device)
        total_loss += loss.item()

    return total_loss / num_batches


# 4. 用已经加载 OpenAI 预训练权重的 gpt 评估两个数据集
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gpt.to(device)
gpt.eval()

torch.manual_seed(123)
with torch.no_grad():
    train_loss = calc_loss_loader(train_loader, gpt, device)
    val_loss = calc_loss_loader(val_loader, gpt, device)

print(f"Training loss: {train_loss:.5f}")
print(f"Validation loss: {val_loss:.5f}")

Training loss: 3.75475
Validation loss: 3.55962


In [ ]:
import torch
import tiktoken
from gpt_download import download_and_load_gpt2
from chapter05 import generate, text_to_token_ids, token_ids_to_text  # 第 5.3 节支持采样的版本

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = tiktoken.get_encoding("gpt2")
prompt = "Every effort moves you"

for model_size in ("124M", "1558M"):
    settings, params = download_and_load_gpt2(
        model_size=model_size,
        models_dir=MODELS_DIR,
    )

    config = GPT_CONFIG_124M.copy()
    config.update(
        context_length=settings["n_ctx"],
        emb_dim=settings["n_embd"],
        n_heads=settings["n_head"],
        n_layers=settings["n_layer"],
        qkv_bias=True,
    )

    comparison_gpt = GPTModel(config)
    load_weights_into_gpt(comparison_gpt, params)
    del params

    comparison_gpt.to(device)
    comparison_gpt.eval()

    torch.manual_seed(123)
    with torch.no_grad():
        token_ids = generate(
            model=comparison_gpt,
            idx=text_to_token_ids(prompt, tokenizer).to(device),
            max_new_tokens=25,
            context_size=config["context_length"],
            top_k=50,
            temperature=1.5,
        )

    print(f"\nGPT-2 {model_size}")
    print(token_ids_to_text(token_ids, tokenizer))
    del comparison_gpt

In [14]:

def generate(model, idx, max_new_tokens, context_size,
             temperature=0.0, top_k=None, eos_id=None):

    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:, -1, :]

        if top_k is not None:
            top_logits, _ = torch.topk(logits, top_k)
            min_val = top_logits[:, -1]
            logits = torch.where(
                logits < min_val,
                torch.tensor(float("-inf")).to(logits.device),
                logits
            )

        if temperature > 0.0:
            logits = logits / temperature
            probs = torch.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
        else:
            idx_next = torch.argmax(logits, dim=-1, keepdim=True)

        if idx_next == eos_id:
            break

        idx = torch.cat((idx, idx_next), dim=1)

    return idx
import tiktoken
from chapter04 import GPTModel
from chapter04 import generate_text_simple
from chapter02 import create_dataloader_v1
tokenizer = tiktoken.get_encoding("gpt2")
def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text, allowed_special = {'<|endoftext|>'})
    encoded_tensor = torch.tensor(encoded).unsqueeze(0)
    return encoded_tensor

def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0)
    return tokenizer.decode(flat.tolist())

torch.manual_seed(123)
token_ids = generate(
    model=gpt,
    idx=text_to_token_ids("Every effort moves you", tokenizer).to(device),
    max_new_tokens=25,
    context_size=NEW_CONFIG["context_length"],
    top_k=50,
    temperature=1.5
)
print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

Output text:
 你好妙製の爆弾の古剑好谭(GuJian)) �


In [17]:
import torch
import tiktoken
from gpt_download import download_and_load_gpt2

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = tiktoken.get_encoding("gpt2")
prompt = "Every effort moves you"

for model_size in ("124M", "1558M"):
    settings, params = download_and_load_gpt2(
        model_size=model_size,
        models_dir=MODELS_DIR,
    )

    config = GPT_CONFIG_124M.copy()
    config.update(
        context_length=settings["n_ctx"],
        emb_dim=settings["n_embd"],
        n_heads=settings["n_head"],
        n_layers=settings["n_layer"],
        qkv_bias=True,
    )

    comparison_gpt = GPTModel(config)
    load_weights_into_gpt(comparison_gpt, params)
    del params

    comparison_gpt.to(device)
    comparison_gpt.eval()

    torch.manual_seed(123)
    with torch.no_grad():
        token_ids = generate(
            model=comparison_gpt,
            idx=text_to_token_ids(prompt, tokenizer).to(device),
            max_new_tokens=25,
            context_size=config["context_length"],
            top_k=50,
            temperature=1.5,
        )

    print(f"\nGPT-2 {model_size}")
    print(token_ids_to_text(token_ids, tokenizer))
    del comparison_gpt

File already exists and is up-to-date: gpt2\124M\checkpoint
File already exists and is up-to-date: gpt2\124M\encoder.json
File already exists and is up-to-date: gpt2\124M\hparams.json
File already exists and is up-to-date: gpt2\124M\model.ckpt.data-00000-of-00001
File already exists and is up-to-date: gpt2\124M\model.ckpt.index
File already exists and is up-to-date: gpt2\124M\model.ckpt.meta
File already exists and is up-to-date: gpt2\124M\vocab.bpe

GPT-2 124M
Every effort moves you toward finding an ideal new way to practice something!

What makes us want to be on top of that?


File already exists and is up-to-date: gpt2\1558M\checkpoint
File already exists and is up-to-date: gpt2\1558M\encoder.json
File already exists and is up-to-date: gpt2\1558M\hparams.json


model.ckpt.data-00000-of-00001:  10%|▉         | 600M/6.23G [36:59<5:47:18, 270kiB/s]  


Download from https://openaipublic.blob.core.windows.net/gpt-2/models/1558M/model.ckpt.data-00000-of-00001 failed. Trying https://huggingface.co/rasbt/gpt2-tensorflow/resolve/main/1558M/model.ckpt.data-00000-of-00001.


model.ckpt.data-00000-of-00001:   6%|▌         | 345M/6.23G [23:03<6:33:30, 249kiB/s]  


Download from https://huggingface.co/rasbt/gpt2-tensorflow/resolve/main/1558M/model.ckpt.data-00000-of-00001 failed. Trying https://f001.backblazeb2.com/file/LLMs-from-scratch/gpt2/1558M/model.ckpt.data-00000-of-00001.


model.ckpt.data-00000-of-00001:   3%|▎         | 174M/6.23G [10:29<6:04:49, 277kiB/s]   


KeyboardInterrupt: 